In [1]:
import json
import numpy as np
import pandas as pd
from scipy import sparse

CARPETA = "../data/e3"

df = pd.read_json(f"{CARPETA}/datos_E3.jsonl", lines=True)
df["fila"] = np.arange(len(df))          # posición en las matrices

tfidf = sparse.load_npz(f"{CARPETA}/tfidf.npz")
extra = np.load(f"{CARPETA}/features_extra.npy")

with open(f"{CARPETA}/clases_tipo.json", encoding="utf-8") as f:
    clases_tipo = json.load(f)
clase_a_num = {c: i for i, c in enumerate(clases_tipo)}

# comprobaciones: todo debe tener el mismo número de filas
print(len(df), tfidf.shape, extra.shape)
print(df.groupby(["corpus", "split"]).size())

# tarea tipo
tipo = df[df["etiqueta_tipo"].notna()]
tipo_train = tipo[tipo.split == "train"].explode("etiqueta_tipo")
tipo_dev   = tipo[tipo.split == "dev"]
tipo_test  = tipo[tipo.split == "test"]

print(len(tipo_train), len(tipo_dev), len(tipo_test))
print(tipo_train["etiqueta_tipo"].value_counts())

7843 (7843, 4503) (7843, 302)
corpus     split
espont     dev       273
           test      266
           train    1301
political  dev       401
           test      413
           train    1870
proto      dev       489
           test      489
           train    2341
dtype: int64
3200 695 750
etiqueta_tipo
ad_hominem                   631
hombre_de_paja               404
falso_dilema                 343
generalizacion_apresurada    342
falsa_causalidad             337
apelacion_emocion            319
falacia_credibilidad         273
ad_populum                   202
deduccion_invalida           174
distraccion                  106
otra_falacia                  69
Name: count, dtype: int64


Xgboost pre optuna


In [ ]:
import os
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import f1_score

tipo_dev  = tipo_dev.assign(etiqueta_tipo=tipo_dev["etiqueta_tipo"].str[0])
tipo_test = tipo_test.assign(etiqueta_tipo=tipo_test["etiqueta_tipo"].str[0])

def preparar(d, matriz):
    X = matriz[d["fila"].values]                     
    y = d["etiqueta_tipo"].map(clase_a_num).values    
    return X, y

X_tr, y_tr = preparar(tipo_train, tfidf)
X_dv, y_dv = preparar(tipo_dev, tfidf)
X_te, y_te = preparar(tipo_test, tfidf)

# mayor cuanto más rara es su clase
pesos = compute_sample_weight("balanced", y_tr)

modelo = XGBClassifier(n_estimators=200, random_state=0, n_jobs=-1)
modelo.fit(X_tr, y_tr, sample_weight=pesos)

# F1 macro en el dev de Political 
mascara = (tipo_dev["corpus"] == "political").values
pred_dev = modelo.predict(X_dv)
print("F1 macro dev Political:", f1_score(y_dv[mascara], pred_dev[mascara], average="macro"))

# CSV de predicciones: dev y test, con el nombre de la clase
def tabla(d, X, y, modelo):
    pred = modelo.predict(X)
    return pd.DataFrame({
        "id": d["id"].values,
        "split": d["split"].values,
        "real": [clases_tipo[i] for i in y],
        "predicho": [clases_tipo[i] for i in pred],
    })

salida = pd.concat([tabla(tipo_dev, X_dv, y_dv, modelo),
                    tabla(tipo_test, X_te, y_te, modelo)])
os.makedirs("predicciones", exist_ok=True)
salida.to_csv("predicciones/tipo__xgboost__tfidf__s0.csv", index=False)
print(salida.head())

F1 macro dev Political: 0.27286682039715165
           id split                       real                   predicho
0  fpr_0008_f   dev  generalizacion_apresurada  generalizacion_apresurada
1  fpr_0010_f   dev  generalizacion_apresurada                 ad_populum
2  fpr_0016_f   dev  generalizacion_apresurada          apelacion_emocion
3  fpr_0019_f   dev  generalizacion_apresurada                 ad_populum
4  fpr_0028_f   dev  generalizacion_apresurada  generalizacion_apresurada


In [3]:
from sklearn.metrics import classification_report

print(pd.Series(y_dv[mascara]).map(lambda i: clases_tipo[i]).value_counts())
print()
print(classification_report(
    y_dv[mascara], pred_dev[mascara],
    labels=range(len(clases_tipo)), target_names=clases_tipo, zero_division=0))

ad_hominem                   61
hombre_de_paja               52
falsa_causalidad             46
apelacion_emocion            40
otra_falacia                 24
falso_dilema                 22
ad_populum                   14
deduccion_invalida            8
falacia_credibilidad          8
distraccion                   7
generalizacion_apresurada     7
Name: count, dtype: int64

                           precision    recall  f1-score   support

               ad_hominem       0.45      0.49      0.47        61
               ad_populum       0.67      0.29      0.40        14
        apelacion_emocion       0.21      0.25      0.23        40
       deduccion_invalida       0.14      0.12      0.13         8
              distraccion       0.00      0.00      0.00         7
     falacia_credibilidad       0.12      0.12      0.12         8
         falsa_causalidad       0.61      0.37      0.46        46
             falso_dilema       0.33      0.55      0.41        22
generalizacion_ap

Optuna TDIDF

In [ ]:
import optuna
import json

optuna.logging.set_verbosity(optuna.logging.WARNING)

def objetivo(trial):
    params = {
        "n_estimators":     trial.suggest_int("n_estimators", 100, 500),
        "max_depth":        trial.suggest_int("max_depth", 3, 10),
        "learning_rate":    trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample":        trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
    }
    m = XGBClassifier(**params, random_state=0, n_jobs=-1)
    m.fit(X_tr, y_tr, sample_weight=pesos)
    pred = m.predict(X_dv[mascara])
    return f1_score(y_dv[mascara], pred, average="macro",
                    labels=range(len(clases_tipo)), zero_division=0)

estudio = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
)
estudio.optimize(objetivo, n_trials=50, show_progress_bar=True)

print(estudio.best_value)
print(estudio.best_params)

  0%|          | 0/50 [00:00<?, ?it/s]

0.3122090245722251
{'n_estimators': 325, 'max_depth': 10, 'learning_rate': 0.010300393839724047, 'subsample': 0.7051436519540372, 'colsample_bytree': 0.3266858561500163, 'min_child_weight': 1}


El nombre lo cambio para tenerlo organizado ya que no merece ejecutar otra vez solo por cambiar el nombre del archivo 

Optuna

In [3]:
import json
import numpy as np
import pandas as pd
import optuna
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import f1_score

optuna.logging.set_verbosity(optuna.logging.WARNING)

# En dev y test la etiqueta es una lista de un elemento: se pasa a texto.
# (Solo si todavía es lista, para que ejecutarla dos veces no la estropee)
if isinstance(tipo_dev["etiqueta_tipo"].iloc[0], list):
    tipo_dev  = tipo_dev.assign(etiqueta_tipo=tipo_dev["etiqueta_tipo"].str[0])
    tipo_test = tipo_test.assign(etiqueta_tipo=tipo_test["etiqueta_tipo"].str[0])

def preparar(d, matriz):
    X = matriz[d["fila"].values]                      # filas de la matriz
    y = d["etiqueta_tipo"].map(clase_a_num).values    # clase -> número
    return X, y

In [4]:
import time

def optimizar_tipo(nombre, matriz, n_trials=50):
    X_tr, y_tr = preparar(tipo_train, matriz)
    X_dv, y_dv = preparar(tipo_dev, matriz)
    pesos = compute_sample_weight("balanced", y_tr)
    mask = (tipo_dev["corpus"] == "political").values

    def objetivo(trial):
        params = {
            "n_estimators":     trial.suggest_int("n_estimators", 100, 500),
            "max_depth":        trial.suggest_int("max_depth", 3, 10),
            "learning_rate":    trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            "subsample":        trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 1.0),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        }
        m = XGBClassifier(**params, random_state=0, n_jobs=-1)
        m.fit(X_tr, y_tr, sample_weight=pesos)
        pred = m.predict(X_dv[mask])
        return f1_score(y_dv[mask], pred, average="macro",
                        labels=range(len(clases_tipo)), zero_division=0)

    t0 = time.time()
    estudio = optuna.create_study(direction="maximize",
                                  sampler=optuna.samplers.TPESampler(seed=42))
    estudio.optimize(objetivo, n_trials=n_trials, show_progress_bar=True)
    print(f"{nombre}: F1 macro dev Political = {estudio.best_value:.3f} "
          f"({(time.time()-t0)/60:.1f} min)")
    print(estudio.best_params)

    with open(f"mejores_params_xgboost_tipo_{nombre}.json", "w") as f:
        json.dump(estudio.best_params, f, indent=2)
    return estudio.best_params

# tipo con vector medio + ? y !  (302 columnas)
optimizar_tipo("w2v_medio", extra)

  0%|          | 0/50 [00:00<?, ?it/s]

w2v_medio: F1 macro dev Political = 0.291 (23.6 min)
{'n_estimators': 284, 'max_depth': 3, 'learning_rate': 0.29568482013918407, 'subsample': 0.8025397429272421, 'colsample_bytree': 0.9667874239112968, 'min_child_weight': 9}


{'n_estimators': 284,
 'max_depth': 3,
 'learning_rate': 0.29568482013918407,
 'subsample': 0.8025397429272421,
 'colsample_bytree': 0.9667874239112968,
 'min_child_weight': 9}

In [6]:
def entrenar_variante(nombre, matriz, train_df, params_de=None, semillas=(0, 1, 2)):
    with open(f"mejores_params_xgboost_tipo_{params_de or nombre}.json") as f:
        mejores = json.load(f)
    y_tr = train_df["etiqueta_tipo"].map(clase_a_num).values
    X_tr = matriz[train_df["fila"].values]
    X_dv, y_dv = preparar(tipo_dev, matriz)
    X_te, y_te = preparar(tipo_test, matriz)

    # Evaluación solo con Political
    m_dv = (tipo_dev["corpus"] == "political").values
    m_te = (tipo_test["corpus"] == "political").values

    presentes = np.unique(y_tr)
    a_local = {c: i for i, c in enumerate(presentes)}
    y_loc = np.array([a_local[c] for c in y_tr])
    pesos = compute_sample_weight("balanced", y_loc)

    f1s = []
    for s in semillas:
        m = XGBClassifier(**mejores, random_state=s, n_jobs=-1)
        m.fit(X_tr, y_loc, sample_weight=pesos)

        pred_dv = presentes[m.predict(X_dv[m_dv])]
        pred_te = presentes[m.predict(X_te[m_te])]
        f1s.append(f1_score(y_dv[m_dv], pred_dv, average="macro",
                            labels=range(len(clases_tipo)), zero_division=0))

        salida = pd.concat([
            pd.DataFrame({"id": tipo_dev["id"].values[m_dv], "split": "dev",
                          "real": [clases_tipo[i] for i in y_dv[m_dv]],
                          "predicho": [clases_tipo[i] for i in pred_dv]}),
            pd.DataFrame({"id": tipo_test["id"].values[m_te], "split": "test",
                          "real": [clases_tipo[i] for i in y_te[m_te]],
                          "predicho": [clases_tipo[i] for i in pred_te]}),
        ])
        salida.to_csv(f"predicciones/tipo__xgboost__{nombre}__s{s}.csv", index=False)

    print(f"{nombre}: F1 macro dev Political = {np.mean(f1s):.3f} ± {np.std(f1s):.3f}  {np.round(f1s, 3)}")

In [8]:
entrenar_variante("w2v_medio", extra, tipo_train)
entrenar_variante("tfidf", tfidf, tipo_train, params_de="tfidf")
entrenar_variante("solo_political", tfidf,
                  tipo_train[tipo_train["corpus"] == "political"], params_de="tfidf")

w2v_medio: F1 macro dev Political = 0.265 ± 0.019  [0.291 0.246 0.26 ]
tfidf: F1 macro dev Political = 0.301 ± 0.009  [0.312 0.291 0.299]
solo_political: F1 macro dev Political = 0.262 ± 0.006  [0.261 0.27  0.255]
